# NTN scene-classification dataset

Forward dataset for classifying an NTN terminal's scene (rural / suburban / urban) from
distilled LSPs. Details: `docs/scene_classification.md`.

* **Classes**: P.1410 block PPP scenes. Medians come from the Chiu & Roy rural/urban wall
  parameters (suburban interpolated), mapped to (α, β, γ) and calibrated so block P_LoS
  matches Chiu eq. (27) (§3).
* **Per scene** (one LHS row): α, β, γ log-uniform ×/÷(1 + p) around the class median;
  street heading U[0°, 90°]; one ITU anchor material per role (ground / wall / roof), drawn
  from class weights, jittered ×/÷1.25 in ε and σ; scattering, lobe width and XPD;
  one polarization pair. Real Starlink orbit per (scene, elevation); 20 UEs (`walk`: 10
  walking UEs, each solved at 11 snapshots 1 s apart, heading U[0°, 360°), speed
  U[1.0, 1.6] m/s, satellite frozen during the walk).
* **Swept**: elevations 10–80°, carriers 2 / 20 GHz.
* **No scene caching**: each runner builds a scene in scratch, solves it, deletes it.

## 1 - Environment and imports

In [ ]:
import os, sys
from pathlib import Path

os.environ['DRJIT_LIBLLVM_PATH'] = '/usr/lib/x86_64-linux-gnu/libLLVM-20.so'

# Repo root: nearest ancestor holding src/utils, from this notebook or the kernel's cwd.
_starts = [Path(p).resolve().parent for p in [globals().get("__vsc_ipynb_file__")] if p]
_starts.append(Path.cwd().resolve())
REPO = next((d for s in _starts for d in (s, *s.parents, s / "Ray_Tracing")
             if (d / "src" / "utils").is_dir()), None)
if REPO is None:
    raise RuntimeError(f"Ray_Tracing repo not found above {[str(s) for s in _starts]}")
if str(REPO.parent) not in sys.path:
    sys.path.insert(0, str(REPO.parent))
print("repo:", REPO)

import json, pickle, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from Ray_Tracing.src.utils import scene_classes as scn
from Ray_Tracing.src.utils import scene_dataset as sd
from Ray_Tracing.src.utils import sweep_utils as su
from Ray_Tracing.src.utils import variant_manifest as vm

## 2 - Run settings

Fixed for every class and dataset. Physics is the sensitivity baseline: depth 3, diffuse +
diffraction, merged shapes, path-domain Doppler. Materials come from the manifests (§4),
so `SCATTERING_COEFF` here is only the fallback.

In [ ]:
SWEEP_ROOT     = str(REPO / "Results" / "scene_class")
GPUS           = (0, 1)
ELEVATIONS     = [10, 20, 30, 40, 50, 60, 70, 80]   # 90 has no orbit draw (peak must exceed E)
FREQS          = {"2GHz": 2e9, "20GHz": 20e9}
N_RX_PER_SCENE = 20
# Every dataset uses one pair: K is not comparable across pairs (opposite elevation trends, see
# report/base_vs_chiu). The LHS still draws cfg.POL_PAIR so scene draws match the mixed-pol
# base_mixedpol run; dataset_variants overwrites it. None restores the drawn pairs.
POL_PAIR       = "RHCP/RHCP"

BASE_CFG = {
    "FREQUENCY": 2e9, "TX_POWER_DBM": 74.0, "SLANT_DIST_M": 600e3,   # fallback only: orbits set TX
    "SCENE_CENTER": [0.0, 0.0], "RX_HEIGHT": 1.5,
    "SCATTERING_COEFF": 0.3, "MAX_DEPTH": 3, "LOS": True, "SPECULAR_REFLECTION": True,
    "DIFFUSE_REFLECTION": True, "DIFFRACTION": True,
    "EDGE_DIFFRACTION": True,    # roof rims: roof and walls are separate meshes, so no shared wedge
    "REFRACTION": False, "BEAMFORMING": True,
    "SCENE_RADIUS": float(np.hypot(scn.PPP_KW["Lx"] / 2, scn.PPP_KW["Ly"] / 2)),
    "BASE_SAMPLES_PER_SRC": 10_000_000, "BASE_MAX_PATHS": 10_000_000,
    "MERGE_SHAPES": True,        # 3 shapes: ~0.7 s solve instead of ~22 s of Dr.Jit codegen
    "RX_CHUNK": 25,
    "RNG_SEED": 42, "CHECKPOINT_INTERVAL": 10, "RAW_SAVE_INTERVAL": 200,
    "N_RX_PER_SCENE": N_RX_PER_SCENE, "PPP_RX_GRID_RES": 2.0, "PPP_RX_MARGIN": 20.0,
    "PPP_RX_CLEARANCE": 1.0, "PPP_VERIFY_EVERY": 0,
    # Baseline pair; scenes override it with POL_PAIR (single-pol solve when all scenes agree).
    "TX_POL_TYPE": "RHCP", "RX_POL_TYPE": "RHCP", "POL_BASE_PATTERN": "iso",
    "IONOSPHERE_ENABLED": True, "VTEC_MEAN_TECU": 10.0, "VTEC_STD_TECU": 5.0, "B_L_TESLA": 2e-5,
    "ATMOSPHERE_ENABLED": True, "ATM_RHO_G_M3": 7.5, "RAIN_R_MM_H": 0.0, "ATM_SCINTILLATION": True,
    # Path-domain Doppler only: the time-domain estimator is kHz-bin leakage at LEO.
    "DOPPLER_ENABLED": True, "DOPPLER_NUM_TIME_STEPS": 1,
    "SOLVER_SEED_OFFSET": 0, "K_FIT_N": 0,
}

## 3 - Classes, per-scene factors and the calibration gate

`(lo, hi, "log")` is log-uniform, `(choices, weights, "cat")` a weighted categorical,
`(lo, hi, "int")` discrete-uniform. Only the geometry medians and the material anchor
weights depend on the class; every other factor has the same range for all classes.

In [ ]:
DATASETS = {   # seed_offset keeps each dataset's scenes disjoint from the others'
    "pilot":  dict(n=30,  perturb=0.20, pooled=False, seed_offset=0,      freqs=["2GHz"]),
    "base":   dict(n=400, perturb=0.20, pooled=False, seed_offset=0,      freqs=["2GHz", "20GHz"]),
    "hard":   dict(n=150, perturb=0.30, pooled=False, seed_offset=20_000, freqs=["2GHz"]),
    "matctl": dict(n=150, perturb=0.20, pooled=True,  seed_offset=40_000, freqs=["2GHz"]),
    # base's scenes (scenes_from: its manifests, first n) with 10 walking UEs; the TX stays at
    # the elevation's orbit draw. UE-side cfg only, so base's solve keys are untouched.
    "walk":   dict(n=400, perturb=0.20, pooled=False, seed_offset=0,      freqs=["2GHz", "20GHz"],
                   scenes_from="base",
                   cfg={"N_RX_PER_SCENE": 10, "UE_TRAJ_STEPS": 11, "UE_TRAJ_DT_S": 1.0,
                        "UE_SPEED_RANGE_MPS": [1.0, 1.6]}),
}
# Smoke test for walk: base's first 3 scenes per class, two elevations (manifests are per dataset).
DATASETS["walk_smoke"] = dict(DATASETS["walk"], n=3, freqs=["2GHz"], elevations=[10, 40])

print(pd.DataFrame(scn.CLASS_MEDIANS).T.assign(kappa=pd.Series(scn.KAPPA)).round(3))
for cls in scn.CLASSES:
    print(f"\n── {cls}")
    for name, spec in scn.class_factors(cls).items():
        if spec[-1] == "cat":
            w = spec[1] if spec[1] is not None else [1 / len(spec[0])] * len(spec[0])
            print(f"  {name:36s} cat  " + ", ".join(f"{c} {p:.2f}" for c, p in zip(spec[0], w)))
        else:
            print(f"  {name:36s} {spec[2] if len(spec) > 2 else 'lin':4s} [{spec[0]:.4g}, {spec[1]:.4g}]")

In [ ]:
# Calibration gate (CPU, ~5 min): block P_LoS at the class medians vs Chiu eq. (27).
# KAPPA in scene_classes was fitted so |diff| <= ~3.5 pts at 10-50 deg.
CAL = pd.DataFrame(scn.calibration_table(n_scenes=20, n_ue=200))
CAL["diff_pts"] = 100 * (CAL.P_los_block - CAL.P_los_chiu)
display(CAL.pivot(index="elevation", columns="scene_class", values="diff_pts").round(1))
fig, ax = plt.subplots(figsize=(6, 3.5))
for cls, g in CAL.groupby("scene_class"):
    l, = ax.plot(g.elevation, g.P_los_block, "o-", label=f"{cls} block")
    ax.plot(g.elevation, g.P_los_chiu, "--", color=l.get_color(), label=f"{cls} Chiu (27)")
ax.set(xlabel="elevation [deg]", ylabel="P_LoS"); ax.legend(fontsize=7); plt.show()
bad = CAL[(CAL.elevation <= 50) & (CAL.diff_pts.abs() > 5)]
assert bad.empty, f"calibration gate failed:\n{bad}"
ordered = CAL.pivot(index="elevation", columns="scene_class", values="P_los_block")
assert ((ordered.rural >= ordered.suburban) & (ordered.suburban >= ordered.urban)).all()

## 4 - Orbit table and class manifests

One real-TLE orbit table covers every seed of every dataset (the orbit draw is
class-blind); it is built once by a CPU pool (~10 s per scene / `ORBIT_PROCS`) and cached.
Manifests only *describe* scenes; they are cached per (dataset, class).

In [ ]:
LAT_MAX, ORBIT_TOL_DEG, ORBIT_PROCS = 60.0, 1.0, 32
CATALOG    = str(REPO / "tle_catalog" / "starlink.tle")
ORBIT_PATH = os.path.join(SWEEP_ROOT, "orbit_table.pkl")

blocks = sorted({(scn.SEED_BLOCK[c] + d["seed_offset"], max(o["n"] for o in DATASETS.values()
                                                             if o["seed_offset"] == d["seed_offset"]))
                 for d in DATASETS.values() for c in scn.CLASSES})
need = {s for b, n in blocks for s in range(b, b + n)}
table = pickle.load(open(ORBIT_PATH, "rb")) if os.path.exists(ORBIT_PATH) else None
if table is None or not need <= set(table["scenes"]) or \
        not set(map(float, ELEVATIONS)) <= set(table["elevations"]):
    os.makedirs(SWEEP_ROOT, exist_ok=True)
    subprocess.run([sys.executable, "-m", "Ray_Tracing.src.utils.orbit_sampling",
                    "--seeds", *[f"{b}:{n}" for b, n in blocks],
                    "--elevations", *[str(e) for e in ELEVATIONS],
                    "--out", ORBIT_PATH, "--catalog", CATALOG, "--procs", str(ORBIT_PROCS),
                    "--seed", "0", "--lat_max", str(LAT_MAX), "--tol", str(ORBIT_TOL_DEG)],
                   cwd=str(REPO.parent), check=True)
    table = pickle.load(open(ORBIT_PATH, "rb"))
print(f"orbit table: {len(table['scenes'])} scenes, elevations {table['elevations']}, "
      f"catalog {table['catalog_sha256'][:12]}")


def manifest_path(ds, cls):
    return os.path.join(SWEEP_ROOT, ds, "_manifests", f"{cls}.pkl")


def dataset_variants(ds):
    o = DATASETS[ds]
    out = []
    for cls in scn.CLASSES:
        p = manifest_path(ds, cls)
        m = vm.load_manifest(p) if os.path.exists(p) else None
        pol = m and {"{TX_POL_TYPE}/{RX_POL_TYPE}".format(**s["cfg_overrides"]) for s in m["scenes"]}
        if m is None and "scenes_from" in o:   # LHS draws depend on n: copy, don't redraw
            m = vm.load_manifest(manifest_path(o["scenes_from"], cls))
            m["scenes"], m["n_scenes"] = m["scenes"][:o["n"]], o["n"]
            vm.save_manifest(m, p)
        if m is None or (POL_PAIR and pol != {POL_PAIR}):
            m = m or scn.class_manifest(cls, o["n"], table, o["perturb"], o["pooled"], o["seed_offset"])
            vm.save_manifest(scn.fixed_pol_manifest(m, POL_PAIR) if POL_PAIR else m, p)
        out += [{"name": f"{tag}__{cls}", "manifest": p, "elevations": list(o.get("elevations", ELEVATIONS)),
                 "cfg_overrides": {"FREQUENCY": FREQS[tag], **o.get("cfg", {})}} for tag in o["freqs"]]
    return out


VARIANTS = {ds: dataset_variants(ds) for ds in DATASETS}
m = vm.load_manifest(manifest_path("base", "urban"))
print(sd.scene_meta(m).drop(columns="scene_seed").describe(include="all").T.iloc[:, :4])

## 5 - Size and cost

Measured on the sensitivity pipeline (RTX 3080, merged shapes, 1e7 rays): ~0.7 s per scene
plus ~0.45 s per UE at 10° and ~0.07 s per UE at 20°+. Per-scene polarization pairs make
both arrays dual-pol; `DUAL_POL_FACTOR` is a guess until the pilot measures it.
These per-UE figures are pessimistic now that `K_FIT_N = 0`: `walk_smoke` (110 receivers per scene)
measured ~6–10 s per scene at both 10° and 40° on one 3080.

In [ ]:
DUAL_POL_FACTOR = 1.0 if POL_PAIR else 1.5   # single-pol solve when every scene shares POL_PAIR


def scene_seconds(e, n_rx=N_RX_PER_SCENE):
    return (0.7 + n_rx * (0.45 if e <= 10 else 0.07)) * DUAL_POL_FACTOR


for ds, o in DATASETS.items():
    c = o.get("cfg", {})
    n_rx = c.get("N_RX_PER_SCENE", N_RX_PER_SCENE) * c.get("UE_TRAJ_STEPS", 1)   # receivers per scene
    s = sum(scene_seconds(e, n_rx) for e in o.get("elevations", ELEVATIONS)) \
        * o["n"] * len(scn.CLASSES) * len(o["freqs"])
    print(f"{ds:7s} {o['n']:4d} scenes/class x {len(o['freqs'])} carrier(s): "
          f"{s / 3600:5.1f} GPU-h  (~{s / 3600 / len(GPUS):.1f} h on {len(GPUS)} GPUs)")

## 6 - Launch

Runs every pending (variant, elevation) of the datasets in `RUN` in the tmux session
`scene_class`; finished pairs are skipped, interrupted ones resume. Two queues on the same
GPUs would run out of memory, so this refuses to start while any `run_stochastic_sim.py`
is alive unless `ALLOW_CONCURRENT`.

In [ ]:
RUN              = ["walk"]   # base, walk_smoke done (2026-10-20). Later ["hard", "matctl"]
RESUME           = True
ALLOW_CONCURRENT = False

busy = subprocess.run(["pgrep", "-f", "run_stochastic_sim.py"], capture_output=True, text=True).stdout.split()
if busy and not ALLOW_CONCURRENT:
    raise RuntimeError(f"run_stochastic_sim.py already running (PIDs {busy}): another queue "
                       "holds the GPUs; wait for it or set ALLOW_CONCURRENT")
entries = []
for ds in RUN:
    d = os.path.join(SWEEP_ROOT, ds)
    n_jobs = sum(len(su.pending_elevations(d, v)) for v in VARIANTS[ds])
    print(f"{ds}: {len(VARIANTS[ds])} variants, {n_jobs} pending (variant, elevation) jobs -> {d}")
    entries += su.queue_entries(d, VARIANTS[ds], BASE_CFG, resume=RESUME)

# What the runners will read: the config.json files on disk, not this kernel's BASE_CFG.
for ds in RUN:
    for v in VARIANTS[ds]:
        c = json.load(open(os.path.join(SWEEP_ROOT, ds, v["name"], "config.json")))
        print(f"  {ds}/{v['name']}: f={c['FREQUENCY']/1e9:g} GHz depth={c['MAX_DEPTH']} "
              f"diffuse={c['DIFFUSE_REFLECTION']} doppler={c['DOPPLER_ENABLED']}"
              f"/{c['DOPPLER_NUM_TIME_STEPS']} n_rx={c['N_RX_PER_SCENE']} "
              f"walk={c.get('UE_TRAJ_STEPS', 1)}x{c.get('UE_TRAJ_DT_S', '-')}s@{c.get('UE_SPEED_RANGE_MPS', '-')}")

su.start_detached(entries, SWEEP_ROOT, gpus=GPUS, session="scene_class")
# Progress: tail -f Results/scene_class/_queue.log, or tmux attach -t scene_class

## 7 - Status and pilot checks

In [ ]:
for ds in RUN:
    print(f"═══ {ds}")
    su.sweep_status(os.path.join(SWEEP_ROOT, ds), VARIANTS[ds])

In [ ]:
# Pilot checks: P_LoS ordered by class, Doppler present, metal roofs solve, all pol pairs present.
L = sd.load_links(os.path.join(SWEEP_ROOT, "pilot"), "pilot")
display(L.pivot_table(index="elevation", columns="scene_class", values="is_los").round(3))
print("doppler_rms_hz NaN/zero fraction:", float((~(L.doppler_rms_hz > 0)).mean()))
print("pol pairs:", L.groupby("pol_pair").scene_seed.nunique().to_dict())
bad = L[~np.isfinite(L.PG_dB)]
print(f"non-finite PG: {len(bad)} links; by roof anchor:", bad.roof_anchor.value_counts().to_dict())
print(L.groupby("roof_anchor")[["PG_dB", "lgDS"]].agg(["mean", "count"]).round(2))